In [1]:
# ============================================================
# CATEGORY A — INSTALL & IMPORTS
# ============================================================
!pip install kagglehub torch torchvision scikit-learn pandas matplotlib

import os
import random
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import r2_score

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models


     |████████████████████████████████| 68 kB 3.0 MB/s eta 0:00:01
     |████████████████████████████████| 73.6 MB 44 kB/s s eta 0:00:01
     |████████████████████████████████| 1.9 MB 9.2 MB/s eta 0:00:01
     |████████████████████████████████| 11.1 MB 6.8 MB/s eta 0:00:01
     |████████████████████████████████| 10.8 MB 12.7 MB/s eta 0:00:01
     |████████████████████████████████| 7.8 MB 14.7 MB/s eta 0:00:01
     |████████████████████████████████| 174 kB 16.0 MB/s eta 0:00:01
     |████████████████████████████████| 78 kB 24.9 MB/s eta 0:00:01
     |████████████████████████████████| 64 kB 14.2 MB/s eta 0:00:01
     |████████████████████████████████| 134 kB 14.5 MB/s eta 0:00:01
     |████████████████████████████████| 6.3 MB 7.2 MB/s eta 0:00:01
     |████████████████████████████████| 200 kB 20.6 MB/s eta 0:00:01
     |████████████████████████████████| 1.6 MB 27.3 MB/s eta 0:00:01     |███▋                            | 184 kB 27.3 MB/s eta 0:00:01
     |████████████████████████████████|

/Users/dong/Desktop/projects/skin_disease/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/dong/Desktop/projects/skin_disease/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Matplotlib is building the font cache; this may take a moment.


In [2]:
# ============================================================
# CATEGORY B — DOWNLOAD DATASETS
# ============================================================
dermnet_path = kagglehub.dataset_download("shubhamgoel27/dermnet")
print("Dermnet path:", dermnet_path)


  4%|▍         | 79.0M/1.72G [00:05<02:07, 13.8MB/s]


KeyboardInterrupt: 

In [ ]:
# ============================================================
# CATEGORY C — BUILD METADATA TABLE (SCAN IMAGES)
# ============================================================
image_rows = []
supported_ext = {".jpg", ".jpeg", ".png"}

for root, dirs, files in os.walk(dermnet_path):
    for f in files:
        if os.path.splitext(f)[-1].lower() in supported_ext:
            full = os.path.join(root, f)
            label = os.path.basename(os.path.dirname(full))
            image_rows.append([full, label])

df = pd.DataFrame(image_rows, columns=["image_path", "label"])
print("Images found:", len(df))


In [ ]:
# ============================================================
# CATEGORY D — ADD SYNTHETIC DEMOGRAPHIC & CLINICAL FEATURES
# ============================================================
np.random.seed(42)

df["age"] = np.random.randint(1, 90, len(df))
df["sex"] = np.random.choice(["M", "F"], len(df))
df["fitz"] = np.random.randint(1, 6, len(df))
df["prev_skin_cancer"] = np.random.choice([0,1], len(df), p=[0.85, 0.15])
df["family_history"]   = np.random.choice([0,1], len(df), p=[0.7, 0.3])
df["outdoor_job"]      = np.random.choice([0,1], len(df), p=[0.8, 0.2])

# Encode labels
label_le = LabelEncoder()
df["label_enc"] = label_le.fit_transform(df["label"])

# Encode sex
sex_le = LabelEncoder()
df["sex_enc"] = sex_le.fit_transform(df["sex"])

df.head()


In [ ]:
# ============================================================
# CATEGORY E — PREPROCESS TABULAR FEATURES
# ============================================================
numeric_cols = ["age", "fitz", "prev_skin_cancer", "family_history", "outdoor_job"]
tabular_cols = numeric_cols + ["sex_enc"]

scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])


In [ ]:
# ============================================================
# CATEGORY F — TRAIN/VALIDATION SPLIT
# ============================================================
train_df, val_df = train_test_split(
    df,
    test_size=0.15,
    stratify=df["label_enc"],
    random_state=42
)


In [ ]:
# ============================================================
# CATEGORY G — DATASET CLASS & IMAGE TRANSFORMS
# ============================================================
img_size = 224

train_tf = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, .456, .406], [0.229, .224, .225])
])

val_tf = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, .456, .406], [0.229, .224, .225])
])


class DermnetDataset(Dataset):
    def __init__(self, df, tabular_cols, transform):
        self.df = df.reset_index(drop=True)
        self.tabular_cols = tabular_cols
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img = Image.open(row["image_path"]).convert("RGB")
        img = self.transform(img)

        tab = torch.tensor(row[self.tabular_cols].values.astype("float32"))
        y   = torch.tensor(row["label_enc"], dtype=torch.float32)

        return img, tab, y


train_loader = DataLoader(DermnetDataset(train_df, tabular_cols, train_tf),
                          batch_size=16, shuffle=True)

val_loader   = DataLoader(DermnetDataset(val_df, tabular_cols, val_tf),
                          batch_size=16, shuffle=False)


In [ ]:
# ============================================================
# CATEGORY H — MULTIMODAL MODEL (IMAGE CNN + TABULAR MLP)
# ============================================================
class MultiSkinNet(nn.Module):
    def __init__(self, num_tab_features):
        super().__init__()

        self.cnn = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        cnn_out = self.cnn.fc.in_features
        self.cnn.fc = nn.Identity()

        self.mlp = nn.Sequential(
            nn.Linear(num_tab_features, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 64),
            nn.ReLU()
        )

        self.fc = nn.Linear(cnn_out + 64, 1)

    def forward(self, img, tab):
        img_f = self.cnn(img)
        tab_f = self.mlp(tab)
        fused = torch.cat([img_f, tab_f], dim=1)
        return self.fc(fused).squeeze(1)


device = "cuda" if torch.cuda.is_available() else "cpu"
model = MultiSkinNet(num_tab_features=len(tabular_cols)).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


In [ ]:
# ============================================================
# CATEGORY I — TRAINING & EVALUATION FUNCTIONS
# ============================================================
def train_epoch():
    model.train()
    losses = []
    for img, tab, y in train_loader:
        img, tab, y = img.to(device), tab.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(img, tab)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return np.mean(losses)


def eval_epoch():
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for img, tab, y in val_loader:
            img, tab = img.to(device), tab.to(device)
            pred = model(img, tab)
            preds.extend(pred.cpu().numpy())
            trues.extend(y.numpy())
    return np.array(preds), np.array(trues)


In [ ]:
# ============================================================
# CATEGORY J — TRAIN MODEL (FAST VERSION)
# ============================================================

# ---- OPTIONAL SPEED-UP: Reduce image size for training ----
# Just change img_size earlier near the transforms:
# img_size = 128  # instead of 224

# ---- OPTIONAL SPEED-UP: Freeze CNN feature extractor ----
for param in model.cnn.parameters():
    param.requires_grad = False

# Only train the tabular MLP + final fusion layer
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(trainable_params, lr=1e-3)

# ---- Training loop ----
EPOCHS = 2  # runs MUCH faster on CPU
for ep in range(1, EPOCHS + 1):
    train_loss = train_epoch()
    print(f"Epoch {ep}: Train Loss = {train_loss:.4f}")

print("Fast training complete!")


In [ ]:
# ============================================================
# CATEGORY K — R² SCORE + PREDICTED VS TRUE PLOT
# ============================================================
preds, trues = eval_epoch()
r2 = r2_score(trues, preds)

print("\nR² SCORE:", r2)

plt.figure(figsize=(6,6))
plt.scatter(trues, preds, alpha=0.4)
plt.xlabel("True Label (numeric)")
plt.ylabel("Predicted Label Score")
plt.title(f"Predicted vs True (R²={r2:.3f})")
plt.plot([trues.min(), trues.max()], [trues.min(), trues.max()], 'r--')
plt.show()
